<a href="https://colab.research.google.com/github/winarwahyuw/data-science-250401020025-winar-wahyu-wulansari/blob/main/Data_Science_Pertemuan_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## PERTEMUAN 10
#### Nama : Winar Wahyu Wulansari
#### NIM : 250401020025
#### Kelas : IF405


In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_samples = 1000
mock_data = {
    'tenure': np.random.randint(1, 72, n_samples),
    'MonthlyCharges': np.random.uniform(20, 120, n_samples),
    'TotalCharges': np.random.uniform(20, 8000, n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples),
    'Churn': np.random.choice(['Yes', 'No'], n_samples, p=[0.265, 0.735])
}
df = pd.DataFrame(mock_data)

print("Dimensi data:", df.shape)
print("\nProporsi kelas Churn:")
print(df["Churn"].value_counts(normalize=True).round(3))

Dimensi data: (1000, 6)

Proporsi kelas Churn:
Churn
No     0.729
Yes    0.271
Name: proportion, dtype: float64


In [2]:

from sklearn.model_selection import train_test_split

df_encoded = pd.get_dummies(df, columns=['Contract', 'InternetService'], drop_first=True)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Preprocessing selesai.")
print(f"Data latih (X_tr): {X_tr.shape}, Data uji (X_te): {X_te.shape}")

Preprocessing selesai.
Data latih (X_tr): (800, 7), Data uji (X_te): (200, 7)


In [3]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_tr, y_tr)

print("Model RandomForestClassifier berhasil dilatih dengan penanganan imbalanced data.")

Model RandomForestClassifier berhasil dilatih dengan penanganan imbalanced data.


In [4]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

print("\nClassification Report:")
print(classification_report(y_te, y_pred, target_names=['Tidak Churn', 'Churn']))
print(f"ROC-AUC Score: {roc_auc_score(y_te, proba):.3f}")


Classification Report:
              precision    recall  f1-score   support

 Tidak Churn       0.72      0.93      0.81       146
       Churn       0.17      0.04      0.06        54

    accuracy                           0.69       200
   macro avg       0.45      0.48      0.44       200
weighted avg       0.57      0.69      0.61       200

ROC-AUC Score: 0.449


In [5]:
print("5 Contoh Probabilitas Churn Pelanggan:")
for i, p in enumerate(proba[:5]):
    print(f"Pelanggan ke-{i+1}: Peluang Churn = {p*100:.1f}%")


5 Contoh Probabilitas Churn Pelanggan:
Pelanggan ke-1: Peluang Churn = 36.3%
Pelanggan ke-2: Peluang Churn = 30.3%
Pelanggan ke-3: Peluang Churn = 34.7%
Pelanggan ke-4: Peluang Churn = 51.0%
Pelanggan ke-5: Peluang Churn = 45.3%


## **Kesimpulan**

*   Model Random Forest gagal mendeteksi pelanggan yang Churn karena data yang digunakan dibuat secara acak (mock data) tanpa korelasi nyata antara fitur dan variabel target.

*   Performa Sangat Rendah: Nilai ROC-AUC berada di angka 0.449 (lebih buruk daripada tebakan acak 0.50) dan Recall kelas Churn hanya 0.04 (hanya berhasil menebak 4% pelanggan yang benar-benar churn).

*   Penyebab Utama: Fitur-fitur seperti tenure dan MonthlyCharges dibuat secara independen terhadap Churn. Karena tidak ada pola statistik dalam data (pure noise), model tidak dapat mempelajari hubungan apa pun.

*   Alur Kode Sudah Tepat: Pemrosesan teknik (one-hot encoding, stratified split, serta class_weight="balanced") secara sintaksis sudah benar. Kegagalan murni disebabkan oleh kualitas data yang acak.


